# Figure 3 — treatment-induced peripheral immune remodeling

Run from top to bottom. The cells follow Figure 3a–l. Every analytical panel exports its input/result table to `resources/Figure3/`.


In [ ]:
# Locate the package regardless of whether Jupyter was started from the project root or notebooks/.
candidates <- c(
  getwd(),
  file.path(getwd(), ".."),
  file.path(getwd(), "figure_2_3_reproduction"),
  file.path(getwd(), "work_path", "figure_2_3_reproduction")
)
package_dir <- candidates[file.exists(file.path(candidates, "functions", "00_setup.R"))][1]
if (is.na(package_dir)) stop("Cannot locate figure_2_3_reproduction. Set package_dir manually.")
package_dir <- normalizePath(package_dir)
options(figure_package_dir = package_dir)
source(file.path(package_dir, "functions", "00_setup.R"))
check_figure_packages()
function_dir <- file.path(package_dir, "functions")

source(file.path(function_dir, '01_palettes.R'))
source(file.path(function_dir, '02_cell_frequency.R'))
source(file.path(function_dir, '04_program_definitions.R'))
source(file.path(function_dir, '05_program_scoring.R'))
source(file.path(function_dir, '06_survival.R'))
source(file.path(function_dir, '07_tcr_startrac.R'))
source(file.path(function_dir, '08_correlations.R'))
source(file.path(function_dir, '09_expression.R'))
source(file.path(function_dir, '10_liver_metastasis.R'))
source(file.path(function_dir, '11_delta_ratio_survival.R'))

seurat_merge <- load_seurat_final()
meta <- seurat_merge@meta.data
clinical_df <- make_clinical_table(meta)
fig_dir <- make_resource_dir('Figure3')


## Fig. 3a — D7 versus D0 immune-cell subset frequencies


In [ ]:
diff_results_fig3a <- calculate_cluster_diff(seurat_merge, group_var = 'sample_type_rn', cell_cluster = 'cell_type3', group1 = 'D7', group2 = 'D0', methods = 'part', paired = FALSE)
fig3a <- plot_cell_frequency_volcano(diff_results_fig3a, 'Fig. 3a: D7 versus D0 immune-cell subset frequencies')
print(fig3a$plot)
save_ggplot_pdf(fig3a$plot, 'Fig3a_D7_vs_D0_cell_frequency.pdf', 'Figure3', 6, 6)
write_resource_table(fig3a$data, 'Fig3a_D7_vs_D0_cell_frequency.csv', 'Figure3')


## Fig. 3b — immune-program changes from D0 to D7


In [ ]:
diff_program_fig3b <- calculate_targeted_sample_diffs(seurat_merge, immune_programs_targeted, patient_col = 'orig.ident', timepoint_col = 'sample_type_rn', subtype_col = 'cell_type3', baseline_value = 'D0', treatment_value = 'D7', assay = 'RNA', slot = 'data', min_cells_per_patient_program = 5, min_genes_present = 3, score_method = 'mean_expr')
fig3b <- plot_program_volcano(diff_program_fig3b$diff_table, title = 'Immune Program Changes: D7 vs D0', diff_cutoff = 0.05, p_cutoff = 0.05, label_celltypes = c('CD4_LAG3', 'CD8_Prolif', 'NK_Prolif', 'NK_XCL1', 'Plasma_cells', 'CD4_Treg', 'DC_cDC2', 'Mono_all'), max_overlaps = 20, label_size = 5)
print(fig3b)
save_ggplot_pdf(fig3b, 'Fig3b_immune_program_change.pdf', 'Figure3', 6, 6)
write_resource_table(diff_program_fig3b$diff_table, 'Fig3b_immune_program_change.csv', 'Figure3')


## Fig. 3c — longitudinal program scores

Confirmed source: `plot_program_boxplot_from_score_res` with the supplied four programs and `AddModuleScore`.


In [ ]:
score_res_all_fig3c <- calculate_targeted_patient_scores_all_samples(seurat_merge, immune_programs_targeted, patient_col = 'patient_id', timepoint_col = 'sample_type_rn', subtype_col = 'cell_type3', sample_col = 'orig.ident', assay = 'RNA', slot = 'data', min_cells_per_patient_program = 5, min_genes_present = 3, score_method = 'AddModuleScore')
box_res <- plot_program_boxplot_from_score_res(score_res = score_res_all_fig3c, programs = c('CD8_all__cytotoxicity', 'CD8_all__activation', 'CD4_LAG3__checkpoint_exhaustion', 'CD4_Treg__suppressive_program'), group_col = 'sample_type_rn', group_values = c('D0', 'D7', 'postICI'), group_labels = c('D0', 'D7', 'postICI'), comparisons = list(c('D0', 'D7'), c('D0', 'postICI'), c('D7', 'postICI')), box_alpha = 1, point_alpha = 1, alternative = 'greater', test_method = 't.test', show_p_adj = FALSE, point_size = 1.5, p_label_type = 'p', facet_ncol = 4, legend_position = 'bottom', title = NULL, subtitle = NULL, y_label = 'Mean Module Score', x_label = 'Treatment Stage', output_pdf = file.path(fig_dir, 'Fig3c_longitudinal_program_scores.pdf'), width = 12.5, height = 5.8)
print(box_res$plot)
write_resource_table(box_res$plot_data, 'Fig3c_longitudinal_program_scores.csv', 'Figure3')
write_resource_table(box_res$stat_data, 'Fig3c_longitudinal_program_statistics.csv', 'Figure3')


## Fig. 3d — cytotoxicity-associated gene expression across time


In [ ]:
cd8_obj <- subset(seurat_merge, subset = cell_type_major == 'CD8')
all_cd8_types <- unique(cd8_obj$cell_type_major)
res_sig_list <- lapply(c('D7', 'postICI'), function(tg) {
  out <- lapply(all_cd8_types, function(ct) {
    x <- tryCatch(get_sample_level_degs_specific(cd8_obj, cell_type_name = ct, cell_type_col = 'cell_type_major', group_col = 'sample_type_rn', comparisons = c(tg, 'D0'), sample_col = 'orig.ident'), error = function(e) NULL)
    if (!is.null(x) && nrow(x)) { x$cell_type <- ct; x$Comparison <- paste0(tg, '_vs_D0') }; x
  })
  dplyr::bind_rows(out)
})
res_sig_fig3d <- dplyr::bind_rows(res_sig_list)
fig3d <- make_fig3d_dotplot(cd8_obj, res_sig_fig3d)
print(fig3d$plot)
save_ggplot_pdf(fig3d$plot, 'Fig3d_cytotoxic_gene_expression.pdf', 'Figure3', 12, 5)
write_resource_table(fig3d$data, 'Fig3d_dotplot_resource.csv', 'Figure3')
write_resource_table(fig3d$significant_genes, 'Fig3d_sample_level_DEG.csv', 'Figure3')


## Fig. 3e — D7 CD4_Treg / CD4_LAG3 correlation


In [ ]:
fig3e <- plot_celltype_correlation(meta = meta |> dplyr::filter(sample_type_rn == 'D7'), cell_type1 = 'CD4_Treg', cell_type2 = 'CD4_LAG3', cell_type_var = 'cell_type3', patient_var = 'orig.ident', group_var = 'lm', method = 'spearman')
print(fig3e$plot)
save_ggplot_pdf(fig3e$plot, 'Fig3e_Treg_vs_LAG3_correlation.pdf', 'Figure3', 6, 5)
write_resource_table(fig3e$data, 'Fig3e_Treg_vs_LAG3_correlation.csv', 'Figure3')


## Fig. 3f — STARTRAC expansion of CD4_LAG3 and CD4_Treg

Confirmed function and supplied plotting code.


In [ ]:
tcr_annotation <- load_tcr_annotation()
tcr_meta <- seurat_merge@meta.data
tcr_meta$tcr_id <- tcr_annotation$combine_seq[match(rownames(tcr_meta), tcr_annotation$barcode_id)]
tcr_meta <- tcr_meta |> dplyr::filter(grepl('^CD4_', cell_type3))
startrac_results <- calc_startrac_expansion(tcr_meta, group_col = 'cell_type3', clone_col = 'tcr_id', patient_col = 'orig.ident') |> dplyr::left_join(tcr_meta |> dplyr::select(orig.ident, sample_type_rn) |> dplyr::distinct(), by = 'orig.ident')
p_fig3f <- ggplot2::ggplot(startrac_results |> dplyr::filter(cell_type3 %in% c('CD4_Treg', 'CD4_LAG3')), ggplot2::aes(x = sample_type_rn, y = STARTRAC_Expa, color = sample_type_rn)) + ggplot2::geom_boxplot(outlier.shape = NA, alpha = 1, width = 0.6) + ggplot2::geom_jitter(width = 0.2, alpha = 1) + ggplot2::facet_wrap(~cell_type3, scales = 'free_y') + ggplot2::scale_color_manual(values = cell_type_colors_updated) + ggpubr::stat_compare_means(comparisons = list(c('D0', 'D7'), c('D7', 'postICI'), c('D0', 'postICI'))) + ggplot2::labs(title = 'CD4 subtype STARTRAC Expansion', y = 'Expansion Index', x = NULL) + ggplot2::theme_bw(base_size = 14) + ggplot2::theme(legend.position = 'none', axis.text.x = ggplot2::element_text(angle = 30, hjust = 1))
print(p_fig3f)
save_ggplot_pdf(p_fig3f, 'Fig3f_STARTRAC_expansion.pdf', 'Figure3', 5.8, 4.8)
write_resource_table(startrac_results, 'Fig3f_STARTRAC_expansion.csv', 'Figure3')


## Fig. 3g — CD4_LAG3 and CD4_Treg abundance by liver-metastasis status


In [ ]:
p_fig3g <- plot_clusters_group(seurat_merge@meta.data, 'lm_group', show_clusters = c('CD4_LAG3', 'CD4_Treg'), color_var = 'lm', cell_cluster = 'cell_type3', line = FALSE, levels = sort(unique(seurat_merge$lm_group)), comparision = list(c('noLM_D0', 'LM_D0'), c('noLM_D7', 'LM_D7')))
print(p_fig3g)
save_ggplot_pdf(p_fig3g, 'Fig3g_liver_metastasis_abundance.pdf', 'Figure3', 7.2, 5.5)
fig3g_table <- meta |> dplyr::filter(cell_type3 %in% c('CD4_LAG3', 'CD4_Treg')) |> dplyr::count(orig.ident, patient_id, lm, lm_group, sample_type_rn, cell_type3, name = 'cell_count')
write_resource_table(fig3g_table, 'Fig3g_liver_metastasis_abundance.csv', 'Figure3')


## Fig. 3h — regulatory-cell abundance versus CD8 cytotoxicity score

User-confirmed source call: D7 CD4_Treg proportion versus the CD8 cytotoxicity gene-set score in CD8 cytotoxicity subtypes.


In [ ]:
fig3h <- plot_geneset_correlation(
  seurat_obj = seurat_merge,
  gene_list = immune_programs_targeted$CD8_all__cytotoxicity$genes,
  gene_set_name = 'CD8 Cytotoxicity',
  subset_col = 'sample_type_rn',
  subset_val = c('D7'),
  prop_cell_type = 'CD4_Treg',
  feature_cell_type = immune_programs_targeted$CD8_all__cytotoxicity$subtypes,
  cell_type_col = 'cell_type3',
  sample_col = 'patient_id',
  color_col = 'lm'
)
print(fig3h$plot)
save_ggplot_pdf(fig3h$plot, 'Fig3h_Treg_CD8_cytotoxicity.pdf', 'Figure3', 6.5, 5)
write_resource_table(fig3h$data, 'Fig3h_Treg_CD8_cytotoxicity_resource.csv', 'Figure3')


## Fig. 3i — D7 Neu_CXCR2 abundance and PFS


In [ ]:
fig3i <- plot_stage_cell_survival(meta = meta, target_cell = 'Neu_CXCR2', target_stage = 'D7', cell_col = 'cell_type3', time_col = 'sample_type_rn', patient_col = 'patient_id', surv_time_col = 'pfs_time', surv_status_col = 'pfs_status')
save_survival_pdf(fig3i, 'Fig3i_Neu_CXCR2_PFS.pdf', 'Figure3')
write_resource_table(fig3i$data, 'Fig3i_Neu_CXCR2_PFS_resource.csv', 'Figure3')


## Fig. 3j–k — CD4_Treg D7/D0 ratio against PFS and OS

Both calls are user-confirmed. Fig. 3k retains the supplied original title text (`CD4_Treg Ratio vs PFS`) while its y-axis uses OS time.


In [ ]:
options(repr.plot.width = 6, repr.plot.height = 6)
fig3k_table <- make_d7_d0_ratio_table(meta, 'CD4_Treg', 'os_time')
fig3k <- plot_ratio_pfs_correlation(
  seurat_merge@meta.data,
  'CD4_Treg',
  cell_type_col = 'cell_type3',
  time_col = 'sample_type_rn',
  patient_col = 'patient_id',
  pfs_col = 'os_time',
  group_col = 'lm',
  death_col = 'is_death_e'
) + ggplot2::ggtitle('CD4_Treg Ratio vs PFS') + ggplot2::labs(y = 'OS Time (Months)')
print(fig3k)
save_ggplot_pdf(fig3k, 'Fig3k_CD4_Treg_ratio_OS.pdf', 'Figure3', 6, 6)
write_resource_table(fig3k_table, 'Fig3k_CD4_Treg_ratio_OS_resource.csv', 'Figure3')


## Fig. 3l — proposed model

Manual artwork; preserved in the full verified manuscript Figure 3 image.
